# CS533 Homework 1 | Part 3: Retrieval Models

**Purpose.** This notebook presents and checks the retrieval experiments on two New York Times collections: **C1 (2005–2014)** and **C2 (2015–2025)**. It follows the assignment sections in order: lexical retrieval (3.1), dense retrieval (3.2), query expansion (3.3), hybrid retrieval (3.4), and efficiency (3.5).

**How to use this notebook.** Run the cells from top to bottom in the existing project environment. They read the official query/collection files and **already saved** retrieval results from `C:\Projects\IR\data`. They do **not** build indexes, train embeddings, or rerun retrieval. All rankings use official document IDs and retain the top 100 results per query. Hyperparameter selection and formal significance tests belong to **Part 4**, not this notebook.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# Update ROOT only if the repository has been moved.
ROOT = Path(r"C:\Projects\IR\CS533_HW1")
DATA = ROOT.parent / "data"
OFFICIAL = DATA / "nyt-tc-2026"
RESULTS = DATA / "part3_results"
INDEXES = DATA / "part3_indexes"
ARTICLES = DATA / "articles_mapped.h5"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.preprocessing import preprocess_text

required = [ROOT, OFFICIAL, RESULTS]
assert all(path.exists() for path in required), "Check ROOT and the external data directory."
pd.set_option("display.max_colwidth", 95)
print("Repository:", ROOT)
print("Saved results:", RESULTS)

Repository: C:\Projects\IR\CS533_HW1
Saved results: C:\Projects\IR\data\part3_results


## 3.1 | Lexical retrieval: TF-IDF and BM25

### Collections and preprocessing

Each collection has its own vocabulary and scoring statistics. We search **P1 known-item queries on C1**, **P2 known-item queries on C2**, and all **32 topic queries on both collections**. The official `docno` identifies each article; `web_url` connects the collection metadata to the article text in the HDF5 source.

The document-to-text joins were completed and checked during indexing, with no missing article text. To keep this notebook lightweight, we inspect the metadata and preprocessing here rather than repeating the full HDF5 join.

In [2]:
collections = {
    "C1": pd.read_csv(OFFICIAL / "collection_P1.tsv", sep="\t"),
    "C2": pd.read_csv(OFFICIAL / "collection_P2.tsv", sep="\t"),
}
known_items = {
    "C1": pd.read_csv(OFFICIAL / "queries_ki_P1.tsv", sep="\t"),
    "C2": pd.read_csv(OFFICIAL / "queries_ki_P2.tsv", sep="\t"),
}
topics = pd.read_csv(OFFICIAL / "topics.tsv", sep="\t")

collection_ids = {name: set(df["docno"]) for name, df in collections.items()}
topic_ids = set(topics["tid"])
expected_queries = {
    name: set(known_items[name]["qid"]) | topic_ids for name in collections
}

overview = pd.DataFrame([
    {
        "Collection": name,
        "Documents": len(collections[name]),
        "Known-item queries": len(known_items[name]),
        "Topic queries": len(topics),
        "Queries per run": len(expected_queries[name]),
    }
    for name in ("C1", "C2")
])
display(overview)

example = known_items["C1"].iloc[0]
print("Example query:", example["query"])
print("Processed tokens:", preprocess_text(example["query"]))

,Collection,Documents,Known-item queries,Topic queries,Queries per run
0,C1,173603,1000,32,1032
1,C2,210664,1000,32,1032


Example query: injuries to key players loom over spurs thunder series
Processed tokens: ['injuries', 'to', 'key', 'players', 'loom', 'over', 'spurs', 'thunder', 'series']


### Ranking methods and saved-output checks

**TF-IDF** uses `TfidfVectorizer` with the shared `preprocess_text` analyzer and `sublinear_tf=True`. The saved document and query vectors use L2 normalization. **BM25** uses $k_1=1.2$ and $b=0.75$, with collection-specific document frequency and document-length statistics:

$$
\operatorname{BM25}(t,d) = \operatorname{idf}(t)\,
\frac{\operatorname{tf}(t,d)(k_1+1)}
{\operatorname{tf}(t,d)+k_1(1-b+b|d|/\overline{|d|})},
\qquad
\operatorname{idf}(t)=\ln\left(1+\frac{N-\operatorname{df}(t)+0.5}{\operatorname{df}(t)+0.5}\right).
$$

The overall query score is the sum of its original term contributions, each with weight 1. Document term repetitions still contribute through $\operatorname{tf}(t,d)$.

Each run should contain **1,032 query IDs**, **100 distinct official document IDs per query**, and **103,200 rows**. The helper below checks these properties, score order, and missing values. It reads one CSV at a time and releases it before checking the next.

In [3]:
def audit_run(filename, collection):
    # Check a saved top-100 run without rerunning retrieval.
    path = RESULTS / filename
    result = {"File": filename, "Collection": collection, "Passed": False, "Issue": ""}
    if not path.is_file():
        result["Issue"] = "File missing"
        return result

    try:
        df = pd.read_csv(path)
        required_columns = ["qid", "docno", "rank", "score"]
        problems = []
        if list(df.columns) != required_columns:
            problems.append("Unexpected CSV columns")
        if len(df) != 103_200:
            problems.append("Incorrect row count")
        if df[required_columns].isna().any().any():
            problems.append("Missing values")
        if not np.isfinite(pd.to_numeric(df["score"], errors="coerce")).all():
            problems.append("Non-finite scores")
        if set(df["qid"]) != expected_queries[collection]:
            problems.append("Incorrect query IDs")
        if not df["docno"].isin(collection_ids[collection]).all():
            problems.append("Document outside official collection")

        by_query = df.groupby("qid", sort=False)
        if not by_query.size().eq(100).all():
            problems.append("Not 100 rows per query")
        if not by_query["docno"].nunique().eq(100).all():
            problems.append("Duplicate document within query")
        if not (by_query["rank"].min().eq(1).all()
                and by_query["rank"].max().eq(100).all()
                and by_query["rank"].nunique().eq(100).all()):
            problems.append("Invalid rank range")

        ordered = df.sort_values(["qid", "rank"])
        if not ordered.groupby("qid")["score"].diff().dropna().le(1e-7).all():
            problems.append("Scores not descending")

        result["Passed"] = not problems
        result["Issue"] = "; ".join(problems)
    except (OSError, ValueError, KeyError, TypeError) as exc:
        result["Issue"] = str(exc)[:160]
    return result


def display_audit(rows, group_columns):
    checked = pd.DataFrame(rows)
    table = checked.groupby(group_columns, as_index=False).agg(
        Passed=("Passed", "sum"), Expected=("Passed", "size")
    )
    display(table)
    failures = checked.loc[~checked["Passed"], ["File", "Issue"]]
    if not failures.empty:
        print("Files needing attention:")
        display(failures)
    return checked


lexical_checks = [
    audit_run(f"{coll.lower()}_{method}.csv", coll)
    for coll in ("C1", "C2")
    for method in ("tfidf", "bm25")
]
lexical_audit = display_audit(lexical_checks, ["Collection"])

,Collection,Passed,Expected
0,C1,2,2
1,C2,2,2


In [4]:
# Compare the top five results for an actual P1 known-item query.
example_qid = known_items["C1"].iloc[0]["qid"]
example_target = known_items["C1"].iloc[0]["docno"]

ranks = []
for method in ("tfidf", "bm25"):
    df = pd.read_csv(RESULTS / f"c1_{method}.csv")
    top = df.loc[df["qid"].eq(example_qid)].nsmallest(5, "rank")
    for entry in top.itertuples(index=False):
        ranks.append({
            "Method": method.upper(), "Rank": entry.rank,
            "Official docno": entry.docno, "Score": entry.score,
            "Correct article": entry.docno == example_target,
        })

print("Query:", known_items["C1"].iloc[0]["query"])
print("Target docno:", example_target)
display(pd.DataFrame(ranks).round(4))

Query: injuries to key players loom over spurs thunder series
Target docno: C1-160665


,Method,Rank,Official docno,Score,Correct article
0,TFIDF,1,C1-160665,0.5379,True
1,TFIDF,2,C1-161538,0.2690,False
2,TFIDF,3,C1-122793,0.2585,False
3,TFIDF,4,C1-123361,0.2520,False
4,TFIDF,5,C1-122885,0.2478,False
5,BM25,1,C1-160665,51.9847,True
6,BM25,2,C1-160874,24.3979,False
7,BM25,3,C1-161538,23.9408,False
8,BM25,4,C1-006500,23.8211,False
9,BM25,5,C1-123361,22.5129,False


### Initial known-item results

A known-item query has one target document. We report **Success@1**, **Success@10**, **Recall@100**, and **MRR@10** as useful checks. These are descriptive results; the complete evaluation and significance testing are developed in Part 4. Because the known-item queries are derived from article titles, strong lexical matching is expected.

In [5]:
def known_item_metrics(run, gold_queries):
    # Evaluate only the official known-item targets, excluding topic queries.
    gold = gold_queries[["qid", "docno"]].rename(columns={"docno": "target"})
    matched = run[["qid", "docno", "rank"]].merge(gold, on="qid", how="inner")
    hits = matched.loc[matched["docno"].eq(matched["target"])]
    target_rank = hits.groupby("qid")["rank"].min()
    ranks = gold["qid"].map(target_rank).fillna(np.inf).to_numpy(dtype=float)
    rr_at_10 = np.divide(1.0, ranks, out=np.zeros_like(ranks), where=ranks <= 10)
    return {
        "Success@1": np.mean(ranks <= 1),
        "Success@10": np.mean(ranks <= 10),
        "Recall@100": np.mean(ranks <= 100),
        "MRR@10": np.mean(rr_at_10),
    }


lexical_metrics = []
for coll in ("C1", "C2"):
    for method in ("tfidf", "bm25"):
        run = pd.read_csv(RESULTS / f"{coll.lower()}_{method}.csv")
        lexical_metrics.append({
            "Collection": coll, "Method": method.upper(),
            **known_item_metrics(run, known_items[coll]),
        })

display(pd.DataFrame(lexical_metrics).round(4))

,Collection,Method,Success@1,Success@10,Recall@100,MRR@10
0,C1,TFIDF,0.657,0.907,0.967,0.7528
1,C1,BM25,0.758,0.935,0.971,0.8293
2,C2,TFIDF,0.536,0.801,0.932,0.6236
3,C2,BM25,0.610,0.839,0.939,0.6855


## 3.2 | Dense retrieval with averaged embeddings

For each document or query $x$, the representation is the term-frequency and IDF-weighted average of available word vectors:

$$
\operatorname{emb}(x)=
\frac{\sum_{t\in x\cap V_M}\operatorname{tf}(t,x)\operatorname{idf}(t)\,\mathbf v_t}
{\sum_{t\in x\cap V_M}\operatorname{tf}(t,x)\operatorname{idf}(t)}.
$$

Here $V_M$ is the embedding model's vocabulary. We use **raw term frequency** and collection-specific IDF. After normalization, documents are ranked by query-document **cosine similarity**. A document with no available word vectors is represented by zero; a query with no embedding receives zero similarities, and such queries are counted.

The four models are **M1** (P1-trained), **M2** (continued on P2), **M2s** (fresh P2 model), and **GN** (pretrained Google News). The Google News subset, extracted on Leylak and loaded with direct preprocessed-token keys, contains **152,145** available vectors. The full 3-million-vector pretrained model is not needed here.

In [6]:
dense_checks = [
    audit_run(f"{coll.lower()}_dense_{model}.csv", coll)
    for coll in ("C1", "C2")
    for model in ("m1", "m2", "m2s", "gn")
]
dense_audit = display_audit(dense_checks, ["Collection"])

# Compare all dense models against BM25 on both known-item test sets.
dense_metrics = []
for coll in ("C1", "C2"):
    for model in ("m1", "m2", "m2s", "gn"):
        file = RESULTS / f"{coll.lower()}_dense_{model}.csv"
        if not file.exists():
            continue
        run = pd.read_csv(file)
        dense_metrics.append({
            "Collection": coll, "Method": f"Dense {model.upper()}",
            **known_item_metrics(run, known_items[coll]),
        })

all_known_metrics = pd.concat([
    pd.DataFrame(lexical_metrics), pd.DataFrame(dense_metrics),
], ignore_index=True)
display(all_known_metrics.round(4))

,Collection,Passed,Expected
0,C1,4,4
1,C2,4,4


,Collection,Method,Success@1,Success@10,Recall@100,MRR@10
0,C1,TFIDF,0.657,0.907,0.967,0.7528
1,C1,BM25,0.758,0.935,0.971,0.8293
2,C2,TFIDF,0.536,0.801,0.932,0.6236
3,C2,BM25,0.610,0.839,0.939,0.6855
4,C1,Dense M1,0.481,0.730,0.881,0.5667
5,C1,Dense M2,0.496,0.730,0.879,0.5734
6,C1,Dense M2S,0.458,0.694,0.845,0.5366
7,C1,Dense GN,0.483,0.728,0.872,0.5636
8,C2,Dense M1,0.326,0.593,0.805,0.4060
9,C2,Dense M2,0.351,0.623,0.843,0.4360


### Query vocabulary coverage (required OOV statistics)

The assignment asks for three measurements **for each embedding model and query set**: the percentage of query tokens absent from the model, the percentage of queries with at least one absent token, and the count of queries with no usable embedding. We load these figures from the completed dense-run summaries without rebuilding any document matrices. Known-item and topic queries are reported separately for C1 and C2.


In [7]:
# Read the OOV statistics reported by the completed dense retrieval jobs.
# Field names can vary across saved summaries, so find the three measurements by meaning.
import re


def _norm_key(value):
    return re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")


def _query_set(value):
    name = _norm_key(value)
    if any(word in name for word in ("known_item", "knownitem", "ki_queries", "ki_query")):
        return "Known-item"
    if any(word in name for word in ("topic", "topical")):
        return "Topics"
    return None


def _metric_type(key):
    name = _norm_key(key)
    if (("no_embedding" in name or "without_embedding" in name or
         "zero_embedding" in name or "missing_embedding" in name) and
            ("quer" in name or name in ("no_embedding", "without_embedding"))):
        return "No embedding"
    if "oov" not in name and "out_of_vocabulary" not in name:
        return None
    if (any(s in name for s in ("fraction", "ratio", "rate", "pct", "percent")) and
            ("quer" in name or "with_oov" in name or "any_oov" in name)):
        return "Queries with OOV"
    if "token" in name and any(s in name for s in ("fraction", "ratio", "rate", "pct", "percent")):
        return "OOV tokens"
    return None


def _extract_oov(summary):
    found = {}
    candidates = []

    def visit(obj, path=(), query_set=None):
        if isinstance(obj, dict):
            # Some formats use records with a separate query_set/query_type label.
            for k in ("query_set", "query_type", "set_name", "query_group"):
                if k in obj and isinstance(obj[k], str):
                    query_set = _query_set(obj[k]) or query_set
            for key, value in obj.items():
                current = _query_set(key) or query_set
                if isinstance(value, (int, float)) and not isinstance(value, bool):
                    kind = _metric_type(key)
                    if kind and current:
                        found[(current, kind)] = float(value)
                    if any(s in _norm_key(key) for s in ("oov", "embedding")):
                        candidates.append((".".join((*path, str(key))), value))
                else:
                    visit(value, (*path, str(key)), current)
        elif isinstance(obj, list):
            for i, value in enumerate(obj):
                visit(value, (*path, str(i)), query_set)

    visit(summary)
    return found, candidates


def _percent(value):
    if value is None:
        return np.nan
    return round(100 * value if 0 <= value <= 1 else value, 2)


oov_rows, oov_missing = [], []
for coll in ("C1", "C2"):
    for model in ("M1", "M2", "M2s", "GN"):
        path = RESULTS / f"dense_{coll.lower()}_{model.lower()}" / "dense_summary.json"
        if not path.exists():
            oov_missing.append(f"Missing summary: {path.name} ({coll}/{model})")
            continue
        with path.open(encoding="utf-8") as f:
            stats, available = _extract_oov(json.load(f))
        for group in ("Known-item", "Topics"):
            token_rate = stats.get((group, "OOV tokens"))
            query_rate = stats.get((group, "Queries with OOV"))
            no_embed = stats.get((group, "No embedding"))
            if any(x is None for x in (token_rate, query_rate, no_embed)):
                oov_missing.append(f"{coll}/{model}/{group}: fields not matched; sample paths: {available[:6]}")
            oov_rows.append({
                "Collection": coll, "Model": model, "Query set": group,
                "OOV tokens (%)": _percent(token_rate),
                "Queries with OOV (%)": _percent(query_rate),
                "No embedding (count)": int(no_embed) if no_embed is not None else np.nan,
            })

oov_table = pd.DataFrame(oov_rows)
display(oov_table)
print(f"OOV rows: {len(oov_table)}/16; missing measurements: {len(oov_missing)}")
if oov_missing:
    print("Check these summary field names before final submission:")
    for problem in oov_missing[:12]:
        print("-", problem)


,Collection,Model,Query set,OOV tokens (%),Queries with OOV (%),No embedding (count)
0,C1,M1,Known-item,3.41,22.7,0
1,C1,M1,Topics,0.00,0.0,0
2,C1,M2,Known-item,3.36,22.3,0
3,C1,M2,Topics,0.00,0.0,0
4,C1,M2s,Known-item,4.74,29.5,0
5,C1,M2s,Topics,0.00,0.0,0
6,C1,GN,Known-item,0.99,7.1,0
7,C1,GN,Topics,0.00,0.0,0
8,C2,M1,Known-item,5.87,25.8,0
9,C2,M1,Topics,0.00,0.0,0


OOV rows: 16/16; missing measurements: 0


**Interpretation.** Dense retrieval can match semantically similar wording without requiring every query token to appear in the article. For title-derived known-item queries, this generalization can also lose exact-match precision. For example, the earlier C1 check gave **MRR@10 = 0.8293 for BM25** and **0.5667 for Dense M1**. This is a meaningful difference in effectiveness, not by itself evidence of incorrect implementation.

The saved dense summaries also record three required **out-of-vocabulary (OOV)** measurements: the proportion of query tokens missing from the model, the proportion of queries containing at least one missing token, and the number of queries with no embedding. These should be considered alongside the effectiveness results in Part 4.

## 3.3 | Embedding-based query expansion

For a query, we compute the centroid of its **L2-normalized, in-vocabulary word vectors**. This is **not** the TF-IDF-weighted document representation from Section 3.2. Among words present in both the current collection vocabulary and the chosen embedding model, we choose the $m$ highest-cosine neighbors of the centroid and exclude the original query words.

Each original query term keeps BM25 weight **1**; an expansion word $e$ receives

$$
w_e=\lambda\cos(\mathbf v_e,\mathbf c).
$$

We test $m\in\{5,10,20\}$ and $\lambda\in\{0.5,1.0\}$ for **M1, M2, and GN** on both collections, producing **36 configurations**. Final parameter selection must use P1 and is handled in Part 4.

In [8]:
qe_checks = [
    {**audit_run(f"{coll.lower()}_qe_{model}_m{m}_l{lam}.csv", coll),
     "Model": model.upper()}
    for coll in ("C1", "C2")
    for model in ("m1", "m2", "gn")
    for m in (5, 10, 20)
    for lam in ("0p5", "1p0")
]
qe_audit = display_audit(qe_checks, ["Collection", "Model"])

,Collection,Model,Passed,Expected
0,C1,GN,6,6
1,C1,M1,6,6
2,C1,M2,6,6
3,C2,GN,6,6
4,C2,M1,6,6
5,C2,M2,6,6


In [9]:
# The same five topic queries are shown for all three models and both collections.
with (RESULTS / "qe_topic_expansions.json").open(encoding="utf-8") as f:
    expansion_data = json.load(f)

expansion_rows = []
expansion_problems = []
for coll in ("C1", "C2"):
    for model in ("M1", "M2", "GN"):
        records = expansion_data["collections"][coll][model]["topics"]
        for item in records:
            neighbors = item["neighbors"]
            original_tokens = set(item["tokens"])
            terms = [entry["term"] for entry in neighbors]
            cosines = [float(entry["cosine"]) for entry in neighbors]
            if (any(term in original_tokens for term in terms)
                    or len(terms) != len(set(terms))
                    or not np.isfinite(cosines).all()
                    or any(a < b - 1e-7 for a, b in zip(cosines, cosines[1:]))):
                expansion_problems.append((coll, model, item["qid"]))
            expansion_rows.append({
                "Collection": coll, "Model": model, "Topic": item["qid"],
                "Query": item["query"],
                "Top 5 neighbors (cosine)": ", ".join(
                    f"{entry['term']} ({entry['cosine']:.3f})"
                    for entry in neighbors[:5]
                ),
            })

for coll in ("C1", "C2"):
    print(f"Top five expansion terms for {coll} (cosine similarities)")
    display(pd.DataFrame(expansion_rows).query("Collection == @coll").reset_index(drop=True))
print("Topic expansion checks passed:", len(expansion_problems) == 0)
if expansion_problems:
    print("Investigate:", expansion_problems)

Top five expansion terms for C1 (cosine similarities)


,Collection,Model,Topic,Query,Top 5 neighbors (cosine)
0,C1,M1,T01,africa,"saharan (0.626), african (0.615), bloemfontein (0.601), africans (0.599), lesotho (0.575)"
1,C1,M1,T02,art,"contemporary (0.732), museum (0.656), heckscher (0.621), galleries (0.615), gallery (0.610)"
2,C1,M1,T04,baseball,"league (0.616), selig (0.579), chass (0.572), leagues (0.565), sluggers (0.552)"
3,C1,M1,T23,politics,"lawmaking (0.539), political (0.526), helpline (0.510), donnybrook (0.508), rightward (0.500)"
4,C1,M1,T27,technology,"technologies (0.633), coders (0.577), tech (0.573), virtualization (0.556), computing (0.554)"
5,C1,M2,T01,africa,"african (0.606), sahel (0.600), pretoria (0.597), bloemfontein (0.588), saharan (0.586)"
6,C1,M2,T02,art,"museum (0.678), galleries (0.666), contemporary (0.660), exhibitions (0.627), gallery (0.624)"
7,C1,M2,T04,baseball,"mlb (0.658), league (0.636), solarte (0.589), leagues (0.587), pitcher (0.587)"
8,C1,M2,T23,politics,"lerer (0.554), tribalism (0.490), crosscurrents (0.479), centrism (0.472), helpline (0.467)"
9,C1,M2,T27,technology,"tech (0.649), technologies (0.638), coders (0.577), generative (0.572), software (0.561)"


Top five expansion terms for C2 (cosine similarities)


,Collection,Model,Topic,Query,Top 5 neighbors (cosine)
0,C2,M1,T01,africa,"saharan (0.626), african (0.615), africans (0.599), lesotho (0.575), uganda (0.573)"
1,C2,M1,T02,art,"contemporary (0.732), museum (0.656), heckscher (0.621), galleries (0.615), gallery (0.610)"
2,C2,M1,T04,baseball,"league (0.616), selig (0.579), leagues (0.565), sluggers (0.552), yankees (0.550)"
3,C2,M1,T23,politics,"lawmaking (0.539), political (0.526), rightward (0.500), polarization (0.500), politicians ..."
4,C2,M1,T27,technology,"technologies (0.633), coders (0.577), tech (0.573), computing (0.554), software (0.553)"
5,C2,M2,T01,africa,"african (0.606), sahel (0.600), pretoria (0.597), saharan (0.586), johannesburg (0.578)"
6,C2,M2,T02,art,"museum (0.678), galleries (0.666), contemporary (0.660), exhibitions (0.627), gallery (0.624)"
7,C2,M2,T04,baseball,"mlb (0.658), league (0.636), solarte (0.589), leagues (0.587), pitcher (0.587)"
8,C2,M2,T23,politics,"lerer (0.554), corasaniti (0.521), tribalism (0.490), crosscurrents (0.479), centrism (0.472)"
9,C2,M2,T27,technology,"tech (0.649), technologies (0.638), coders (0.577), generative (0.572), software (0.561)"


Topic expansion checks passed: True


In [10]:
# A fixed example is a diagnostic comparison, not hyperparameter tuning.
qe_example = []
for coll in ("C1", "C2"):
    for name, filename in [
        ("BM25", f"{coll.lower()}_bm25.csv"),
        ("QE M1 (m=5, λ=0.5)", f"{coll.lower()}_qe_m1_m5_l0p5.csv"),
    ]:
        run = pd.read_csv(RESULTS / filename)
        qe_example.append({
            "Collection": coll, "Method": name,
            **known_item_metrics(run, known_items[coll]),
        })
display(pd.DataFrame(qe_example).round(4))

,Collection,Method,Success@1,Success@10,Recall@100,MRR@10
0,C1,BM25,0.758,0.935,0.971,0.8293
1,C1,"QE M1 (m=5, λ=0.5)",0.754,0.935,0.972,0.8277
2,C2,BM25,0.610,0.839,0.939,0.6855
3,C2,"QE M1 (m=5, λ=0.5)",0.606,0.832,0.940,0.6801


**Interpretation.** Query expansion can improve recall by introducing semantically related vocabulary, but additional terms can also displace the correct article in a known-item ranking. In the previously checked C1 example, **BM25 MRR@10 was 0.8293** and **QE M1 ($m=5,\lambda=0.5$) was 0.8277**; Recall@100 rose slightly from **0.971** to **0.972**. This is one configuration, not a conclusion about which setting is best. The report will present the five topic examples for each model, with results retained separately for C1 and C2.

## 3.4 | Hybrid retrieval: linear fusion and RRF

For each query, the candidate set is the **union** of BM25's top 100 and dense retrieval's top 100. This union allows semantically relevant documents found by dense retrieval to enter the final ranking even when BM25 did not rank them in its top 100.

For **linear fusion**, we compute **both actual scores** for every candidate, even if the document was absent from one ranker's original top 100. Each score type is independently min-max normalized over the complete union (or set to zero when all scores tie):

$$
S_\alpha(q,d)=\alpha\,\widehat{s}_{\rm BM25}(q,d)+(1-\alpha)\,\widehat{s}_{\rm emb}(q,d),
\qquad \alpha\in\{0,0.1,\ldots,1\}.
$$

For **reciprocal rank fusion (RRF)**, we instead use original top-100 ranks, and a missing ranker contributes **zero**:

$$
S_{\rm RRF}(q,d)=\sum_{r\in\{{\rm BM25},{\rm emb}\}:d\in\operatorname{top100}_r}\frac{1}{60+\operatorname{rank}_r(d)}.
$$

We test **M1, M2, and GN** against BM25 on C1 and C2. This yields **66 linear-fusion runs** and **6 RRF runs**. Alpha tuning belongs in Part 4.

In [11]:
# Audit the 72 expected hybrid outputs without recomputing model scores.
hybrid_checks = []
for coll in ("C1", "C2"):
    for model in ("m1", "m2", "gn"):
        for tenth in range(11):
            alpha_code = f"{tenth // 10}p{tenth % 10}"
            filename = f"{coll.lower()}_hybrid_{model}_a{alpha_code}.csv"
            hybrid_checks.append({
                **audit_run(filename, coll), "Model": model.upper(), "Fusion": "Linear",
            })
        filename = f"{coll.lower()}_rrf_{model}.csv"
        hybrid_checks.append({
            **audit_run(filename, coll), "Model": model.upper(), "Fusion": "RRF",
        })

hybrid_audit = display_audit(hybrid_checks, ["Collection", "Model", "Fusion"])
print("Total valid hybrid files:", int(hybrid_audit["Passed"].sum()), "/", len(hybrid_audit))

,Collection,Model,Fusion,Passed,Expected
0,C1,GN,Linear,11,11
1,C1,GN,RRF,1,1
2,C1,M1,Linear,11,11
3,C1,M1,RRF,1,1
4,C1,M2,Linear,11,11
5,C1,M2,RRF,1,1
6,C2,GN,Linear,11,11
7,C2,GN,RRF,1,1
8,C2,M1,Linear,11,11
9,C2,M1,RRF,1,1


Total valid hybrid files: 72 / 72


In [12]:
# Endpoint check: α=1 agrees with BM25 ranking; α=0 preserves the dense top-100 set.
# Dense scores can have nearly tied values, so α=0 order need not be bit-for-bit identical.
endpoint_rows = []
for coll in ("C1", "C2"):
    bm_file = RESULTS / f"{coll.lower()}_bm25.csv"
    bm = pd.read_csv(bm_file, usecols=["qid", "docno", "rank"])
    bm_order = bm.sort_values(["qid", "rank"])[["qid", "docno"]].reset_index(drop=True)
    for model in ("m1", "m2", "gn"):
        a1 = RESULTS / f"{coll.lower()}_hybrid_{model}_a1p0.csv"
        a0 = RESULTS / f"{coll.lower()}_hybrid_{model}_a0p0.csv"
        dense_file = RESULTS / f"{coll.lower()}_dense_{model}.csv"
        if not all(path.exists() for path in (a1, a0, dense_file)):
            endpoint_rows.append({"Collection": coll, "Model": model.upper(),
                                  "α=1 BM25 order": "Missing file", "α=0 dense set": "Missing file"})
            continue
        linear1 = pd.read_csv(a1, usecols=["qid", "docno", "rank"])
        linear0 = pd.read_csv(a0, usecols=["qid", "docno"])
        dense = pd.read_csv(dense_file, usecols=["qid", "docno"])
        same_bm25_order = bm_order.equals(
            linear1.sort_values(["qid", "rank"])[["qid", "docno"]].reset_index(drop=True)
        )
        same_dense_set = (set(map(tuple, linear0.itertuples(index=False, name=None)))
                          == set(map(tuple, dense.itertuples(index=False, name=None))))
        endpoint_rows.append({"Collection": coll, "Model": model.upper(),
                              "α=1 BM25 order": same_bm25_order,
                              "α=0 dense set": same_dense_set})

display(pd.DataFrame(endpoint_rows))

,Collection,Model,α=1 BM25 order,α=0 dense set
0,C1,M1,True,True
1,C1,M2,True,True
2,C1,GN,True,True
3,C2,M1,True,True
4,C2,M2,True,True
5,C2,GN,True,True


## 3.5 | Efficiency

We compare indexing or representation-construction time, the **actual dense document matrix storage**, and average query latency. The timings below are the **recorded measurements from completed runs**, not new benchmarks executed by this notebook. Results depend on the machine and software environment.

**Measurement boundaries matter:** lexical indexing includes the collection preparation and sparse weights; dense indexing measures **document embedding construction using cached IDF**. A dense matrix uses $N\times300\times4$ bytes (`float32`), which is **198.67 MiB for C1** and **241.09 MiB for C2**. This matrix storage is not the same as peak process RAM.

In [15]:
# Previously recorded runs: indexing seconds and average query milliseconds.
recorded = [
    ("C1", "TF-IDF", 65.74, 8.84), ("C2", "TF-IDF", 74.80, 13.18),
    ("C1", "BM25", 68.44, 8.14),   ("C2", "BM25", 71.88, 11.68),
    ("C1", "Dense M1", 69.12, 4.61),  ("C2", "Dense M1", 73.18, 5.30),
    ("C1", "Dense M2", 70.06, 4.61),  ("C2", "Dense M2", 73.61, 5.88),
    ("C1", "Dense M2s", 75.87, 4.96), ("C2", "Dense M2s", 79.85, 5.86),
    ("C1", "Dense GN", 77.13, 4.82),  ("C2", "Dense GN", 77.89, 5.68),
]

efficiency = pd.DataFrame(recorded, columns=[
    "Collection", "Method", "Indexing (s)", "Avg. query (ms)",
])
efficiency["Dense matrix (MiB)"] = efficiency.apply(
    lambda row: (len(collections[row["Collection"]]) * 300 * 4 / 2**20)
    if row["Method"].startswith("Dense") else np.nan,
    axis=1,
)
display(efficiency.round(2))

,Collection,Method,Indexing (s),Avg. query (ms),Dense matrix (MiB)
0,C1,TF-IDF,65.74,8.84,NaN
1,C2,TF-IDF,74.80,13.18,NaN
2,C1,BM25,68.44,8.14,NaN
3,C2,BM25,71.88,11.68,NaN
4,C1,Dense M1,69.12,4.61,198.67
5,C2,Dense M1,73.18,5.30,241.09
6,C1,Dense M2,70.06,4.61,198.67
7,C2,Dense M2,73.61,5.88,241.09
8,C1,Dense M2s,75.87,4.96,198.67
9,C2,Dense M2s,79.85,5.86,241.09


**Additional cost for query expansion and hybrid retrieval.** The QE means below are reported across each model-collection pair's six parameter settings; they **exclude one-time nearest-neighbor generation**. Hybrid measurements are for **candidate scoring, normalization, fusion, and ranking**. They exclude the original BM25 and dense first-stage searches, so they must **not** be labeled end-to-end hybrid latency. The job logs and JSON summaries remain the authoritative sources for the detailed measurements.

In [14]:
# Query expansion: previously recorded average query latency across six settings.
# One-time expansion-neighbor generation is excluded and was measured separately.
qe_times = pd.DataFrame([
    ("C1", "M1", 7.56, 570), ("C2", "M1", 10.83, 600),
    ("C1", "M2", 10.59, 579), ("C2", "M2", 11.38, 612),
    ("C1", "GN", 8.24, 741), ("C2", "GN", 10.83, 775),
], columns=["Collection", "QE model", "Avg. QE query (ms)", "Peak process RAM (MiB)"])
display(qe_times)

# Hybrid: display one summary row per model and collection, not 72 diagnostic fields.
# These values cover candidate scoring, normalization, fusion and ranking only.
hybrid_json = RESULTS / "hybrid_clarification_validation.json"
if hybrid_json.exists():
    with hybrid_json.open(encoding="utf-8") as f:
        hybrid_data = json.load(f)

    def _numeric(value):
        return float(value) if isinstance(value, (int, float)) and not isinstance(value, bool) else np.nan

    def _find_rrf_ms(pair):
        for key in ("mean_rrf_query_ms", "rrf_mean_query_ms", "rrf_query_ms", "mean_rrf_ms"):
            if isinstance(pair.get(key), (int, float)):
                return float(pair[key])
        for nested_key in ("rrf_latency", "rrf", "rrf_timing"):
            nested = pair.get(nested_key)
            if isinstance(nested, dict):
                for key in ("mean_query_ms", "mean_latency_ms", "mean_rrf_query_ms"):
                    if isinstance(nested.get(key), (int, float)):
                        return float(nested[key])
        return np.nan

    hybrid_rows = []
    for i, pair in enumerate(hybrid_data.get("pairs", [])):
        collection = pair.get("collection", pair.get("collection_name", f"pair[{i}]"))
        model = pair.get("model", pair.get("embedding_model", "unknown"))
        alpha_records = pair.get("latency_by_alpha", [])
        alpha_times = [x.get("mean_query_ms") for x in alpha_records
                       if isinstance(x, dict) and isinstance(x.get("mean_query_ms"), (int, float))]
        linear_ms = pair.get("mean_linear_query_ms")
        if not isinstance(linear_ms, (int, float)) and alpha_times:
            linear_ms = float(np.mean(alpha_times))
        hybrid_rows.append({
            "Collection": collection,
            "Dense model": model,
            "Linear fusion (ms/query)": _numeric(linear_ms),
            "RRF (ms/query)": _find_rrf_ms(pair),
        })

    hybrid_times = pd.DataFrame(hybrid_rows)
    if not hybrid_times.empty:
        display(hybrid_times.round(3))
        print("Hybrid model-collection pairs:", len(hybrid_times), "/ 6")
        if hybrid_times["RRF (ms/query)"].isna().any():
            print("RRF latency was not identified in this JSON. Inspect saved timing fields if needed.")
    else:
        print("No model-collection pairs were found in hybrid_clarification_validation.json.")
else:
    print("Hybrid summary not found; no hybrid timing values were assumed.")

print("Hybrid timings above EXCLUDE the original BM25 and dense first-stage searches.")


,Collection,QE model,Avg. QE query (ms),Peak process RAM (MiB)
0,C1,M1,7.56,570
1,C2,M1,10.83,600
2,C1,M2,10.59,579
3,C2,M2,11.38,612
4,C1,GN,8.24,741
5,C2,GN,10.83,775


,Collection,Dense model,Linear fusion (ms/query),RRF (ms/query)
0,C1,M1,2.457,NaN
1,C2,M1,1.986,NaN
2,C1,M2,2.298,NaN
3,C2,M2,1.774,NaN
4,C1,GN,2.240,NaN
5,C2,GN,1.858,NaN


Hybrid model-collection pairs: 6 / 6
RRF latency was not identified in this JSON. Inspect saved timing fields if needed.
Hybrid timings above EXCLUDE the original BM25 and dense first-stage searches.


## Summary and next steps

This notebook checks the saved **four lexical**, **eight dense**, **36 query-expansion**, and **72 expected hybrid** retrieval files. The audit tables explicitly flag missing or invalid files; a successful unit test elsewhere is not a substitute for these checks. The examples show how exact-term, semantic, expanded, and fused retrieval differ without altering any earlier results.

**Part 4** will evaluate all models on known-item and topic judgments, tune $m$ and $\lambda$ and the hybrid $\alpha$ **using P1**, apply those settings unchanged to P2, and carry out the required significance and temporal analyses. The final report will provide a compact explanation of the findings and the experimental limitations.